The notebook extracts informations from a corpus to create a knowledge graph.

In [1]:
import json
import ollama
from sentence_transformers import SentenceTransformer
import os
import hashlib
import sys
sys.path.insert(1, "src/graph/")
from graph_builder import get_node_id, extract_graph, compute_chunk_embeddings, merge_graphs, validate_graph, build_neo4j_graph, feed_global_report
sys.path.insert(1, "src/preprocessing/")
from speeches import load_speeches, split_into_chunks, load_template_json, load_prompt_template, prompt_builder

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [2]:
corpus = []
folder = "discours-presidents"
example_json = load_template_json(
    os.path.join(
        "src", 
        "graph",
        "prepare_llm_output_graph_production.json"
    )
)

prompt_template = load_prompt_template(
    os.path.join(
        "src",
        "prompts",
        "prompt_RAG_production.txt"
    )
)

merged_graph = {
    "entities": [],
    "relations": []
}

In [3]:
global_report = {
    "speeches": 0,
    "chunks": 0,
    "entities": 0,
    "relations": 0,
    "duplicate_entities": 0,
    "duplicate_relations": 0,
    "missing_targets": {},
    "missing_sources": {}
}

In [ ]:
corpus = corpus[:1]

In [4]:
# extract data from speech and sotres it into a list of dict with this fields : 
# 'id', 'date', 'title', 'filename', 'header', 'text', 'chunks'
corpus = load_speeches(folder)
for speech in corpus[:1]:
    # split text in chuncks and store it
    speech["chunks"] = split_into_chunks(speech)
    speech["chunks"] = compute_chunk_embeddings(speech["chunks"])
    # label and relations for the speech
    merged_graph = {
    "entities": [],
    "relations": []
    }
    # split speech and made embedding
    for chunk in speech["chunks"]:
            prompt = prompt_builder(prompt_template, example_json, chunk["text"])
            graph = extract_graph(prompt, model = "qwen2.5:7b")
            if graph is None:
                continue
            merged_graph = merge_graphs(
                merged_graph,
                graph
            )
    # check duplicate and no valide elements
    merged_graph, report = validate_graph(merged_graph)
    global_report = feed_global_report(report, speech, global_report)
    # build neo4j_graph
    neo4j_graph = build_neo4j_graph(
        speech,
        merged_graph
    )

In [7]:
global_report

{'speeches': 2,
 'chunks': 43,
 'entities': 156,
 'relations': 124,
 'duplicate_entities': 0,
 'duplicate_relations': 0,
 'missing_targets': {'Dialogue': 1,
  'asséchement des richesses culturelles du monde': 1,
  'atteinte au droit élémentaire des enfants à vivre protégés des turpitudes de certains adultes': 1,
  'atteinte à notre sécurité et donc à notre liberté et à notre intégrité': 1,
  'richesses culturelles du monde': 1,
  'Coopération internationale': 1,
  'Olaf Scholz': 1,
  'Paris': 1,
  'Standardisation': 1,
  'Stocks privés': 1},
 'missing_sources': {'Person': 1, 'Person-1': 1}}

In [6]:
merged_graph

{'entities': [{'type': 'Person', 'name': 'Emmanuel Macron'},
  {'type': 'Theme', 'name': 'Énergie'},
  {'type': 'Theme', 'name': 'Matières premières'},
  {'type': 'Theme', 'name': 'Ressources naturelles'},
  {'type': 'Theme', 'name': 'Industries émergentes'},
  {'type': 'Theme', 'name': 'Pénurie de matières premières'},
  {'type': 'Theme', 'name': 'Croissance économique'},
  {'type': 'Theme', 'name': 'Guerres de la faim'},
  {'type': 'Organization', 'name': 'G20'},
  {'type': 'Person', 'name': 'Monsieur le Président'},
  {'type': 'Theme', 'name': 'Croissance durable'},
  {'type': 'Theme', 'name': 'Régulation'},
  {'type': 'Theme', 'name': 'Développement économique'},
  {'type': 'Person', 'name': 'Monsieur le Président BARROSO'},
  {'type': 'Person', 'name': 'José Manuel'},
  {'type': 'Theme', 'name': 'Consensus'},
  {'type': 'Theme', 'name': 'Présidence française'},
  {'type': 'Theme', 'name': 'Sujet'},
  {'type': 'Theme', 'name': 'Rôle de la présidence'},
  {'type': 'Theme', 'name': '

In [3]:
speech.keys()

dict_keys(['id', 'date', 'title', 'filename', 'header', 'text', 'chunks'])

In [ ]:
for chunk in speech["chunks"]:
        graph = extract_graph(chunk["text"])

# Prompt pour Ollama

In [ ]:
example_json = load_template_json(
    os.path.join(
        "src", 
        "graph",
        "prepare_llm_output_graph_production.json"
    )
)

prompt_template = load_prompt_template(
    os.path.join(
        "src",
        "prompts",
        "prompt_RAG_production.txt"
    )
)

In [ ]:
def extract_graph(prompt,
                  model = "qwen2.5:7b"):

    # prompt = prompt_builder(prompt_template, example_json, text)

    response = ollama.chat(
        model = model,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        format="json",
        options={
            "temperature": 0
        }
    )
    content = response["message"]["content"]
    try:
        graph = json.loads(content)
    except json.JSONDecodeError:
        print("Erreur JSON")
        print(content)
        return None
    return graph

In [ ]:
i = 0

text = corpus[i]['text']

for chunk in speech["chunks"]:
        prompt = prompt_builder(prompt_template, example_json, chunk["text"])
        graph = extract_graph(prompt, model = "qwen2.5:7b")
        break


In [ ]:
graph

# Appel au LLM

In [ ]:
response = ollama.chat(
    model="qwen2.5:7b",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ],
    format="json",
    options={
        "temperature": 0
    }
)

In [ ]:
content = response["message"]["content"]

print(response)

In [ ]:
content = response["message"]["content"]

data = json.loads(content)

In [ ]:
print(data)

The LLM extracts the information. Now we need to add the ids

In [ ]:
# 1. create IDs
for entity in data["entities"]:
    entity["id"] = get_node_id(entity)

# 2. Build index
entity_index = {
    entity["name"]: entity
    for entity in data["entities"]
}

# 3. Relationship
for relation in data["relations"]:
    relation["source_id"] = get_node_id(entity_index[relation["source"]])
    relation["target_id"] = get_node_id(entity_index[relation["target"]])

In [ ]:
graph = {
    "nodes": [],
    "relationships": []
}

for entity in data["entities"]:
    graph["nodes"].append({
        "id": entity["id"],
        "label": entity["type"],      # Label Neo4j
        "properties": {
            "name": entity["name"]
        }
    })

for relation in data["relations"]:
    graph["relationships"].append({
        "type": relation["relation"],
        "source": relation["source_id"],
        "target": relation["target_id"],
        "properties": {
            "evidence": relation["evidence"],
            "confidence": relation["confidence"]
        }
    })

In [ ]:
graph

# Load embedding model

Next step is to add embedding to the node. By this way we can make text similarity querry 

In [ ]:
embedding_model = SentenceTransformer("BAAI/bge-m3")

In [ ]:
EMBEDDING_LABELS = {"Theme", "Event", "Speech", "Chunk"}

for node in graph["nodes"]:
    if node["label"] in EMBEDDING_LABELS:
        embedding = embedding_model.encode(node["properties"]["name"])
        node["properties"]["embedding"] = embedding.tolist()

In [ ]:
graph

In [ ]:
embedding_model = SentenceTransformer("BAAI/bge-m3")
text = """
Le président Emmanuel Macron a rencontré Olaf Scholz à Berlin
afin de renforcer la coopération européenne sur les questions énergétiques.
"""

# Generate embedding
embedding = embedding_model.encode(text).tolist()
print(f"Dimension de l'embedding : {len(embedding)}")